# GeometricNearestNeighbors usage examples

Anton Antonov   
June 2026

---

## Introduction

This notebook has the most typical workflow of using the Python package "GeometricNearestNeighbors", [AAp1].

---

## Setup

In [1]:
from GeometricNearestNeighborsProcessor import *
from RandomDataGenerators import *
from OutlierIdentifiers import *

import numpy
import random

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

---

## Usage examples

Generate random points:

In [3]:
dfPoints = random_data_frame(n_rows=30, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints.shape)
print(dfPoints[1:6])

(30, 2)
          X         Y
1  1.935190  1.174910
2  1.780630  0.202334
3 -0.301526 -1.756783
4 -1.197077  0.018142
5 -0.900910  2.687235


Here is a summary:

In [4]:
dfPoints.describe()

,X,Y
count,30.000000,30.000000
mean,-0.032158,-0.026114
std,1.090938,0.995646
min,-2.187378,-2.222312
25%,-0.898581,-0.492634
50%,-0.201003,0.018348
75%,0.728257,0.605713
max,2.110100,2.687235


Here is a plot of the points:

In [5]:
fig = px.scatter(dfPoints, x="X", y="Y", template="plotly_dark")
fig.show()

A typical pipeline of geometric nearest neighbors processing:

In [6]:
gnnObj = (GeometricNearestNeighborsProcessor(dfPoints)
   .make_nearest_function(distance_function = "EuclideanDistance")
   .compute_thresholds(number_of_nearest_neighbors = 10, aggregation_function = "mean", outlier_identifier = "QuartileIdentifierParameters")
   .find_anomalies()
   .echo_function_value("Anomaly points:")
)

Anomaly points:
          X         Y    Radius
0 -2.187378  0.592765  1.342965
1  1.935190  1.174910  1.368525
2 -0.900910  2.687235  1.975935
3  2.110100 -0.665214  1.306263
4  0.738654 -2.222312  1.642887


One way to plot the anomalies together with the data points using the package "plotly" is to merge into one data frame and add an indicator column:

In [7]:
# Assign data
df1 = gnnObj.take_data()
df2 = gnnObj.take_value()

# Mark points that are in df2
df1['highlight'] = df1.merge(df2.assign(flag=1), on=['X', 'Y'], how='left')['flag'].fillna(0)

# Convert to category for plotting
df1['highlight'] = df1['highlight'].map({0: 'data', 1: 'anomalies'})

# Plot
fig = px.scatter(df1, x='X', y='Y', color='highlight',
                 color_discrete_map={'data': 'blue', 'anomalies': 'red'},
                 template="plotly_dark")

fig.show()

Alternatively, the anomaly points can be shown as smaller, in red, and on top of the data points:

In [8]:
fig = go.Figure()

# All points (larger)
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='data'
))

# Anomaly points (smaller, on top)
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=6, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

Here we generate another set of random points using the same random point generators:

In [9]:
dfPoints2 = random_data_frame(n_rows=40, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints2.shape)

(40, 2)


Here the points of second set are classified into being anomalous or not:

In [20]:
gnnObj.classify(dfPoints2).take_value()

,Index,Radius,Label
0,1,0.603751,True
1,2,0.632141,True
2,3,1.125992,True
3,4,1.249162,False
4,5,0.664255,True
5,6,1.002581,True
6,7,0.668519,True
7,8,0.739559,True
8,9,1.318085,False
9,10,0.924426,True


Plot the original data points (in gray) and the new data points by marking the anomalous ones:

In [21]:
df0 = gnnObj.take_data()

dfClassified = gnnObj.classify(dfPoints2).take_value()
dfCombined = dfPoints2.join(dfClassified)

df1 = dfCombined[dfCombined["Label"] == True]
df2 = dfCombined[dfCombined["Label"] == False]

fig = go.Figure()

# Original data points
fig.add_trace(go.Scatter(
    x=df0['X'],
    y=df0['Y'],
    mode='markers',
    marker=dict(size=8, color='gray'),
    name='data'
))

# Non-anomaly points
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=10, color='blue', line=dict(width=1, color='black')),
    name='non-anomalies'
))

# Anomaly points
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=10, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

---

## References

[AAp1] Anton Antonov, [GeometricNearestNeighborsProcessor](https://github.com/antononcube/Python-GeometricNearestNeighborsProcessor), Python package, (2026), [GitHub/antononcube](https://github.com/antononcube).([PIPy.org](https://pypi.org/project/GeometricNearestNeighborsProcessor/).)

[AAp2] Anton Antonov, [RandomDataGenerators](https://github.com/antononcube/Python-packages/tree/main/RandomDataGenerators), Python package, (2021-2026), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/RandomDataGenerators/).)

[AAp3] Anton Antonov, [OutlierIdentifiers](https://github.com/antononcube/Python-packages/tree/main/OutlierIdentifiers), Python package, (2024), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/OutlierIdentifiers/).)